In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import nltk
import re
from sklearn.datasets import fetch_20newsgroups # Import the 20 newsgroups dataset from the original website.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from collections import Counter
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from collections import Counter

## Loading The Data
We start by loading the 20 newsgroups dataset's raw text data and insepct it to see what are we going to work with.

In [2]:
# Load the data as is from the 20 newsgroups dataset.
data_train_raw = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)

Let's have a look at two randomly chosen example documents of the original raw data.

In [3]:
# Save some example documents (chosen randomly, can change the index to see other documents).
example_document1 = data_train_raw.data[39]
example_document2 = data_train_raw.data[40]

print('1st Example:\n',example_document1)
print('2nd Example:\n',example_document2)

1st Example:
 From: bressler@iftccu.ca.boeing.com (Rick Bressler)
Subject: Re: Gun Lovers (was Re: My Gun is like my American Express Card)
Organization: Boeing Commercial Airplane Group
Lines: 104

/ iftccu:talk.politics.guns / vincent@cad.gatech.edu (Vincent Fox) / 10:34 am  Apr 14, 1993 /

This isn't rec.guns, so maybe this is getting a bet technical, but I
can't resist....

> - A revolver also has the advantage that if it misfires you just pull
>   the trigger again.

Sometimes.....  Depends on WHY it misfired....

> - A double-action revolver (almost all of them) can be hand-cocked first,
>   but will fire merely by pulling the trigger.

I can't imagine doing much combat type shooting single action.....

> - A misfire in a revolver merely means you must pull the trigger again
>   to rotate to the next round.

Assuming the cylinder WILL rotate....

> - A revolver can be carried with the 6th chamber empty and under the
>   hammer for maximum safety, but still can be drawn and fired 

By examining the examples of the original raw data, We noticed that headers, footers, and quoted text contain a lot of irrelevant information that could mislead the classifier or information. Headers often include metadata such as sender names, email addresses, and server details, which do not contribute to the actual topic of discussion. Footers usually contain signatures, which may introduce bias, and quoted text repeats parts of previous messages, creating redundancy. Keeping these elements could cause the model to overfit to superficial patterns rather than meaningful content. 
To ensure the classifier learns from the content of the document itself, we will use the remove parameter to exclude these elements when loading the training and test data.

In [ ]:
# Load the 20 Newsgroups dataset's training set with headers, footers, and quotes removed.
data_train = fetch_20newsgroups(subset='train', 
                                shuffle=True, 
                                random_state=42,
                                remove =("headers","footers","quotes"),
                               )

# Load the 20 Newsgroups dataset's testing set with headers, footers, and quotes removed.
data_test = fetch_20newsgroups(subset='test',
                               shuffle=True,
                               random_state=42,
                               remove=("headers","footers","quotes"),
                              )

# Get the raw documents and labels.
X_train, y_train = data_train.data, data_train.target
X_test, y_test = data_test.data, data_test.target

# Target names (class labels):
target_names = data_train.target_names
print('We can see that our 18,846 samples are divided as follows:')
print('Training data samples -',len(X_train))
print('Testing data samples -',len(X_test))
print('Number of classes -',len(target_names))

## Data distribution
It's interesting to see how does the data is distributed to different classes, and how does the raw data looks like.

In [ ]:
# Count the number of files in each class for the training data
train_class_counts = Counter(y_train)

# Create a bar plot to visualize the file counts per class
sorted_counts = sorted(train_class_counts.items(), key=lambda x: x[1], reverse=True)
classes = [target_names[idx] for idx, _ in sorted_counts]
counts = [count for _, count in sorted_counts]

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=counts, y=classes, hue=classes, palette="viridis", dodge=False)
plt.xlabel("Number of Files", fontsize=12)
plt.ylabel("Class Name", fontsize=12)
plt.title("Training Data: Number of Files per Class", fontsize=14)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()  # Adjust layout so that long class names fit
plt.show()

The data is quite balanced in the most part, However there are some classes("talk.religion.misc" (377 files), "talk.politics.misc" (465 files) and "alt.etheism" (480 files)) that have fewer files compared to the other classes which are around 600 files each. This might lead to higher accuracy on majority classes while underpreforming on minority classes when using Algorithms like Logistic Regression and SVM which optimize a global loss. same thing might happen with kNN and Decision Trees. 
We will see about that later in the project.

## Preprocessing
We can see that our original raw data contains a lot of noise such as punctuation, numbers and special characters. 
As Tf-idf vectorizer will represent the data as a sparse matrix, if we wont remove the noise factors, they will add unnecessary dimensions that do not contribute to meaning and will increase computational cost without meaningful gains.
Also, extra noise can cause models to learn irrelevant patterns, making them less effective.
Another thing is that the data contains stop words(e.g., "the", "is", "and") which are common across all the data, meaning they do not help in distinguishing between categories so keeping them increase noise and will overshadow more meaningful words.

Concluding our wanted Preprocessing steps to execute:
1. Cleaning - Convert text to lowercase (treat "Apple" and "apple" as the same word) and remove punctuation, numbers and special characters.
4. Stopwords Removal - we will filter out common stopwords.
3. Tokenization - Using tf-idf vectorized we will convert the raw data into vectors.
5. Stemming - converting words to their base form, Changing Changed Change will turn into Chang.

In [48]:


# import nltk

# def preprocess_text(text):
#     # Convert text to lowercase. 
#     text = text.lower()
#     # Remove non-word characters (punctuation, symbols, etc.).
#     text = re.sub(r'\W+', ' ', text)
#     # Tokenize the text.
#     tokens = word_tokenize(text)
#     # Initialize the PorterStemmer.
#     stemmer = nltk.PorterStemmer()
#     # Apply stemming to each token.
#     stemmed_tokens = [stemmer.stem(token) for token in tokens]
#     # Rejoin tokens into a single string.
#     return " ".join(stemmed_tokens)
    
# # Create a TF-IDF Vectorizer with the custom preprocessor and stopword removal.
# tfidf_vect = TfidfVectorizer(preprocessor=preprocess_text,
#                              stop_words='english',
#                              max_df=0.5,  # Ignore terms that appear in more than 50% of documents
#                              min_df=5)     # Ignore terms that appear in fewer than 5 documents

import re
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

def custom_analyzer(text):
    # 1. Cleaning & Lowercasing: Convert text to lowercase and remove non-word characters.
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)
    
    # 2. Tokenization: Tokenize the cleaned text.
    tokens = word_tokenize(text)
    
    # tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    
    # 4. Stemming: Convert tokens to their base form using PorterStemmer.
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    
    return stemmed_tokens

# Create a TF-IDF Vectorizer with the custom analyzer.
tfidf_vect = TfidfVectorizer(
    analyzer=custom_analyzer,  # use our custom analyzer function
    stop_words='english',      # 3. Stopwords Removal: Filter out common English stopwords.
    max_df=0.5,                # ignore terms that appear in more than 50% of the documents
    min_df=5                   # ignore terms that appear in fewer than 5 documents
)
# Fit and transform the training data, then transform the testing data.
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

print("TF-IDF vectorized train data shape:", X_train_tfidf.shape)

# doc_index = 0
# doc_tfidf_vec = X_train_tfidf[doc_index] 
# doc_tfidf_arr = doc_tfidf_vec.toarray().flatten()

# tfidf_features = tfidf_vect.get_feature_names_out()
# top_n = 10 
# top_indices = np.argsort(doc_tfidf_arr)[-top_n:][::-1]
# top_words = [tfidf_features[i] for i in top_indices] 
# top_scores = doc_tfidf_arr[top_indices]

# plt.figure(figsize=(10, 6)) 
# plt.barh(top_words[::-1], top_scores[::-1], color='skyblue') 
# plt.xlabel("TF-IDF Score") 
# plt.title("Top 10 TF-IDF Features for a Sample Document") 
# plt.show()

C:\Users\benam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_extraction\text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


TF-IDF vectorized train data shape: (11314, 13368)


## Train and Evaluate Multiple Classifiers
We will train four classifiers:
- Logistic Regression
- Decision Tree
- K-Nearest Neighbors (KNN)
- Support Vector Machine (SVM)

### Confusion Matrix Visualization
Here we display the confusion matrix for the Logistic Regression classifier. This will help us see if there are any patterns in the classification errors.

In [ ]:
# # Define classifiers in a dictionary
# classifiers = {
#     "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
#     "Decision Tree": DecisionTreeClassifier(random_state=42),
#     "KNN": KNeighborsClassifier(), # We go with default k=5
#     "SVM": SVC(kernel='linear', random_state=42)
# }

# # Dictionaries to store accuracy results and confusion matrices
# results = {}
# conf_matrices = {}

# # Loop through each classifier to train and evaluate
# for name, clf in classifiers.items():
#     print(f"\nTraining {name}...")
#     clf.fit(X_train_tfidf, y_train)  # Train classifier on training data
#     y_pred = clf.predict(X_test_tfidf)  # Predict on test data

#     # Calculate and store accuracy
#     acc = accuracy_score(y_test, y_pred)
#     results[name] = acc

#     # Print accuracy and classification report
#     print(f"Accuracy for {name}: {acc:.4f}")
#     print(classification_report(y_test, y_pred, target_names=target_names))

#     # Save confusion matrix for Logistic Regression for later analysis
#     if name == "Logistic Regression":
#         conf_matrices[name] = confusion_matrix(y_test, y_pred)

# # Plot classifier accuracies as a bar chart
# plt.figure(figsize=(8, 5))
# names = list(results.keys())
# acc_values = list(results.values())
# sns.barplot(x=names, y=acc_values,hue=names,dodge=False,legend=False, palette="viridis")
# plt.ylabel("Accuracy")
# plt.ylim(0, 1)
# plt.title("Classifier Accuracy Comparison")
# plt.xticks(rotation=45)
# plt.show()

# # Retrieve the confusion matrix for Logistic Regression and plot it
# cm = conf_matrices["Logistic Regression"]

# plt.figure(figsize=(10, 8))
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
#             xticklabels=target_names, yticklabels=target_names)
# plt.xlabel("Predicted Label")
# plt.ylabel("True Label")
# plt.title("Confusion Matrix: Logistic Regression")
# plt.xticks(rotation=45)
# plt.yticks(rotation=0)
# plt.show()

### Analysis of Confusion Matrix and Next Steps
From the confusion matrix we can observe which classes are often confused with one another. For example, if you see a block of high misclassification between two similar topics, that could be an indication that the classifier struggles with subtle differences in language between those topics.

To gain more insight into the classification decisions, we will now inspect the top 3 words that contribute the most (i.e. have the highest weight) to each class in the Logistic Regression classifier.

### Extracting and Visualizing Top 3 Words per Class
Using the coefficients from the Logistic Regression model (which has one weight per feature per class), we extract for each class the top 3 features (words) with the highest positive weights. We then visualize them.

In [ ]:
# log_reg = classifiers["Logistic Regression"]

# coefs = log_reg.coef_
# feature_names = tfidf_vect.get_feature_names_out()
# top_words_per_class = {}  # dictionary: key = class name, value = (words, weights)

# for idx, class_name in enumerate(target_names):
#     coef = coefs[idx]  # Get indices of the top 3 coefficients (largest positive weights)
#     top3_idx = np.argsort(coef)[-3:]
#     top3_words = feature_names[top3_idx]
#     top3_weights = coef[top3_idx]
#     top_words_per_class[class_name] = (top3_words, top3_weights)

# n_classes = len(target_names)
# ncols = 4
# nrows = (n_classes + ncols - 1) // ncols  # compute rows needed

# fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 4*nrows))
# axes = axes.flatten()

# for i, class_name in enumerate(target_names):
#     words, weights = top_words_per_class[class_name]
#     sns.barplot(x=weights, y=words,hue=weights, ax=axes[i], palette="magma")
#     axes[i].set_title(f"Top Words for Class: {class_name}")
#     axes[i].set_xlabel("Coefficient Weight")
#     axes[i].set_ylabel("")

# for j in range(i+1, len(axes)):
#     fig.delaxes(axes[j])

# plt.tight_layout()
# plt.show()